In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from scipy.stats import zscore
from scipy import sparse

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

### raw number of differential peaks

In [ ]:
peaks_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/atac/peaks/chromhmm/justin.chromhmm.dedup.bed.gz'
peaks = pd.read_csv(peaks_path, sep='\t', header=None)
peaks.index = [f'{x}_{y}' for x,y in zip(peaks[0], peaks[1])]
peaks.shape

In [ ]:
peaks.head()

In [ ]:
fail_peaks = peaks[peaks[13] == 'Fail2']
fail_peaks.shape

In [ ]:
fail_peaks.head()

In [ ]:
fail2_clusts = [f'Fail2_{x}' for x in range(1,6)]
fail2_dist = pd.DataFrame(index=fail2_clusts)
fail2_dist['diff_peaks'] = fail_peaks[12].value_counts()
fail2_dist = fail2_dist.fillna(0)
fail2_dist = fail2_dist.astype(int)
fail2_dist

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
fail2_dist.to_csv(f'{outdir}/fail2_diff_peak_dist.csv', sep='\t')

### zscore

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/atac/peaks/clusters/justin_chromhmm_subcluster.h5ad')

In [ ]:
adata

In [ ]:
adata.X

In [ ]:
# # norm_zscore = cpm.T.apply(zscore).fillna(0)
# norm_zscore.shape

In [ ]:
X = adata.X

libsize = X.sum(axis=1)
libsize[libsize == 0] = 1
X = X / libsize[:, None] * 1e6
mean = X.mean(axis=0)
std = X.std(axis=0)
std[std == 0] = 1
norm_zscore = (X - mean) / std
adata.X = norm_zscore

In [ ]:
adata.var.index = [f'{x}_{y}' for x,y in zip(adata.var['seqnames'], adata.var['start'])]

In [ ]:
tmp_clusts = ['Start_1', 'Start_2',
                   'Sendai_1', 'Sendai_2',
                   'Delayed_1', 'Delayed_2', 'Delayed_3',
                  'Fail1_1', 'Fail1_2', 'Fail1_3', 'Fail1_4',
                  'Inter1_1', 'Inter1_2', 'Inter1_3', 'Inter1_4','Inter1_5',
                  'Inter2_1', 'Inter2_2', 'Inter2_3', 'Inter2_4',
                  'Fail2_1', 'Fail2_2', 'Fail2_3', 'Fail2_4', 'Fail2_5',
                    'IPS_1', 'IPS_2', 'IPS_3',]

In [ ]:
plot_clusts = ['Inter1_1', 'Inter1_2', 'Inter1_3', 'Inter1_4','Inter1_5',
                  'Inter2_1', 'Inter2_2', 'Inter2_3', 'Inter2_4',
                  'Fail2_1', 'Fail2_2', 'Fail2_3', 'Fail2_4', 'Fail2_5',
                    'IPS_1', 'IPS_2', 'IPS_3']
final_clusts = ['Inter1_1', 'Inter1_2', 'Inter1_3', 'Inter1_4','Inter1_5',
                  'Inter2_1', 'Inter2_2', 'Inter2_3', 'Inter2_4',
                  'Fail2_2', 'Fail2_3', 'Fail2_4',
                    'IPS_1', 'IPS_2', 'IPS_3']

In [ ]:
annots = peaks[14].unique()
annots

In [ ]:
tmp_peaks = peaks[peaks[14] == 'Enhancer']
tmp_peaks[12].value_counts().sort_index()

In [ ]:
a = 'Enhancer'
annot_peaks = peaks[peaks[14] == a]
peak_results = []
for s in plot_clusts:
    tmp_peaks = annot_peaks[annot_peaks[12] == s]
    mask = adata.var.index.isin(tmp_peaks.index)
    tmp_adata = adata[:,mask]
    peak_results.append(tmp_adata.X.mean(axis=1))

peak_df = pd.DataFrame(np.vstack(peak_results))
peak_df.columns = tmp_clusts
peak_df.index = plot_clusts
peak_df = peak_df.dropna()
peak_df = peak_df.loc[:,plot_clusts]
peak_df.shape

In [ ]:
peak_df.head()

In [ ]:
fig, ax = plt.subplots(1, figsize=(10, 7))
sns.heatmap(peak_df, vmin=-2, vmax=2, cmap='coolwarm', ax=ax)
ax.set_title(a)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=12)
ax.grid(False)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
peak_df.to_csv(f'{outdir}/fail2_atac_zscore.csv', sep='\t')

In [ ]:
!date